# Rag_pipeline_**Member1**

In [ ]:
# ── Upload WikiBio dataset to Colab runtime ───────────────────
#
from google.colab import files
import shutil, os

if not os.path.exists("wikibio_gpt4o.json"):
    print("📤 Please upload wikibio_gpt4o.json ...")
    uploaded = files.upload()
    print("✅ Uploaded:", list(uploaded.keys()))
else:
    print("✅ wikibio_gpt4o.json already present")

embedding generation and storing it to faiss DB

In [ ]:
import json
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from groq import Groq
from datasets import load_dataset

groq_client = Groq(api_key=GROQ_API_KEY)

# ── Global Setup ──────────────────────────────────────────────
print("🔢 Loading embedding model (multi-qa-MiniLM-L6-cos-v1)...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/multi-qa-MiniLM-L6-cos-v1",
    model_kwargs={"device": "cpu"}
)
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
BATCH = 500

# ══════════════════════════════════════════════════════════════
#  DATABASE 1: SQuAD (General Knowledge)
# ══════════════════════════════════════════════════════════════
print("\n📥 Loading SQuAD dataset from HuggingFace...")
squad = load_dataset("squad", split="validation")

# We extract 200 unique contexts to keep the index fast and lightweight for the demo
unique_squad_contexts = list(set(squad["context"]))[:200]
squad_docs = []

for ctx in unique_squad_contexts:
    for chunk in splitter.split_text(ctx):
        squad_docs.append(Document(page_content=chunk, metadata={"source": "squad"}))

print(f"📄 {len(squad_docs)} SQuAD chunks created.")
print("🏗️ Building SQuAD FAISS index...")

squad_vectorstore = None
for i in range(0, len(squad_docs), BATCH):
    batch = squad_docs[i:i+BATCH]
    if squad_vectorstore is None:
        squad_vectorstore = FAISS.from_documents(batch, embeddings)
    else:
        squad_vectorstore.merge_from(FAISS.from_documents(batch, embeddings))
    print(f"  Processed {min(i+BATCH, len(squad_docs))}/{len(squad_docs)} SQuAD chunks")

squad_vectorstore.save_local("squad_faiss_index")
print("✅ SQuAD FAISS index saved to 'squad_faiss_index'")


# ══════════════════════════════════════════════════════════════
#  DATABASE 2: WikiBio (Biographies)
# ══════════════════════════════════════════════════════════════
print("\n📥 Loading WikiBio GPT-4o dataset...")
try:
    with open("wikibio_gpt4o.json", "r") as f:
        wikibio_data = json.load(f)
    print(f"✅ Loaded {len(wikibio_data)} WikiBio items")

    wikibio_docs = []
    for item in wikibio_data:
        # Use the first 5 reference samples per item as retrieval corpus
        for sample in item["text_samples"][:5]:
            for chunk in splitter.split_text(sample):
                wikibio_docs.append(Document(
                    page_content=chunk,
                    metadata={"unique_id": item.get("unique_id", ""), "source": "wikibio"}
                ))

    print(f"📄 {len(wikibio_docs)} WikiBio chunks created.")
    print("🏗️ Building WikiBio FAISS index...")

    wikibio_vectorstore = None
    for i in range(0, len(wikibio_docs), BATCH):
        batch = wikibio_docs[i:i+BATCH]
        if wikibio_vectorstore is None:
            wikibio_vectorstore = FAISS.from_documents(batch, embeddings)
        else:
            wikibio_vectorstore.merge_from(FAISS.from_documents(batch, embeddings))
        print(f"  Processed {min(i+BATCH, len(wikibio_docs))}/{len(wikibio_docs)} WikiBio chunks")

    wikibio_vectorstore.save_local("wikibio_faiss_index")
    print("✅ WikiBio FAISS index saved to 'wikibio_faiss_index'")

except FileNotFoundError:
    print("⚠️ ERROR: 'wikibio_gpt4o.json' not found. Please upload it to your Colab files to build the WikiBio index.")


def retriever_agent(question: str, k: int = 5) -> str:
    results = vectorstore.similarity_search(question, k=k)
    return "\n\n".join(f"[Passage {i+1}]\n{d.page_content}" for i, d in enumerate(results))


def generator_agent(question: str, context: str) -> str:
    resp = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": """You are a helpful answering agent.
Answer the question completely and concisely.
Use the provided context as your primary source."""},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"}
        ],
        temperature=0.7,
        max_tokens=200
    )
    return resp.choices[0].message.content.strip()


print("✅ Member 1 module ready")